**Openai API Fine-tuning**

In [ ]:
!pip install openai

In [ ]:
import os
import json
from google.colab import userdata
from openai import OpenAI

OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "what is a championes trophy in cricket how it is different from worldcup."
        }
    ]
)

print(completion.choices[0].message.content)

In [ ]:
with open("/content/data.jsonl","r",encoding="utf-8") as file:
  data=[json.loads(line) for line in file]

data, data[0]

In [ ]:
client = OpenAI()

client.files.create(
    file=open("/content/data.jsonl","rb"),
    purpose="fine-tune"
)

files = client.files.list()
for file in files:
  print(file.id)
  print(100*"=")
  print(file.purpose)
  print(100*"=")
  print(file)

In [ ]:
training_file_id="file-PsuEt1x4gLPqY8f4sDD49t"
suffix_name="first finetune model"
client.fine_tuning.jobs.create(
    training_file=training_file_id,
    model="gpt-4.1-nano",
    suffix=suffix_name
)

client.fine_tuning.jobs.list()

In [ ]:
for job in client.fine_tuning.jobs.list():
  print(job.fine_tuned_model)

In [ ]:
for job in client.fine_tuning.jobs.list():
  print(job.fine_tuned_model)

In [ ]:
client.chat.completions.create(
    model="ft:gpt-3.5-turbo-0125:personal:first-finetune-model:B48axNSg",
    messages=[
        {"role": "system", "content": "You are a customer support agent for a smartphone company whose primary goal is to help users with issues they are experiencing with their smartphones. You are friendly and concise. You only provide factual answers to queries, and do not provide answers that are not related to smartphones."},
        {
            "role": "user",
            "content": "What warranty does a smartphone come with?"
        }
    ]
)

In [ ]:
ans=client.chat.completions.create(
    model="ft:gpt-3.5-turbo-0125:personal:first-finetune-model:B48axNSg",
    messages=[
        {"role": "system", "content": "You are a customer support agent for a smartphone company whose primary goal is to help users with issues they are experiencing with their smartphones. You are friendly and concise. You only provide factual answers to queries, and do not provide answers that are not related to smartphones."},
        {
            "role": "user",
            "content": "hi how are you?"
        }
    ]
)

print(ans.choices[0].message.content)

### **Gemini Fine-Tuning**

In [ ]:
!pip install --upgrade google-genai

In [ ]:
import time
from google import genai
from google.genai.types import HttpOptions, CreateTuningJobConfig, TuningDataset
from google.colab import auth
from google.cloud import aiplatform

In [ ]:
PROJECT_ID = "gen-lang-client-0486212375"
LOCATION = "us-central1"
auth.authenticate_user()

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1beta1")
)

resp = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="What is a color of the water?",
)

print(resp.text)

In [ ]:
tuning_job = client.tunings.tune(
    base_model="gemini-2.5-flash",
    training_dataset=training_dataset,
    config=CreateTuningJobConfig(
        tuned_model_display_name="example-sft-job",
    ),
)

print("Tuning Job:", tuning_job.name)

In [ ]:
jobs = client.tunings.list()
for j in jobs:
  print(j.name)

In [ ]:
tuning_job_name = jobs[0].name
tuning_job = client.tunings.get(name=tuning_job_name)

response = client.models.generate_content(
    model=tuning_job.tuned_model.endpoint,
    contents="Why is the sky blue?"
)

print("STATE:", tuning_job.state)
print("MODEL:", tuning_job.tuned_model.model)
print("ENDPOINT:", tuning_job.tuned_model.endpoint)
print(response.text)

In [ ]:
running_states = {"JOB_STATE_PENDING", "JOB_STATE_RUNNING"}

while tuning_job.state in running_states:
  print("State:", tuning_job.state)
  tuning_job = client.tunings.get(name=tuning_job_name)
  time.sleep(60)

print("Final State:", tuning_job.state)
print("MODEL:", tuning_job.tuned_model.model)
print("ENDPOINT:", tuning_job.tuned_model.endpoint)

In [ ]:
client_v1 = genai.Client(
    vertexai=False,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=HttpOptions(api_version="v1")
)

response = client_v1.models.generate_content(
    model=tuning_job.tuned_model.endpoint,
    contents="Why is the sky blue?",
)
print(response.text)

In [ ]:
aiplatform.init(project=PROJECT_ID, location=LOCATION)

MODEL_ID = tuning_job.tuned_model.model
model = aiplatform.Model(model_name=MODEL_ID)
model.delete()
print("Deleted model:", MODEL_ID)

### **SLM Fine-Tuning**

In [ ]:
!pip install -U torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install -U unsloth
!pip install -U transformers==4.56.2 datasets==4.3.0
!pip install -U --no-deps trl==0.22.2

In [ ]:
import torch
from dataclasses import dataclass
from typing import Optional, List
from datasets import load_dataset
from unsloth import FastLanguageModel
from peft import PeftModel
from trl import SFTTrainer, SFTConfig

assert torch.cuda.is_available()

@dataclass
class FineTuneConfig:
  model_name: str
  load_in_4bit: bool = True
  max_seq_length: int = 4096
  dtype: Optional[str] = None

  dataset_name: str = "tatsu-lab/alpaca"
  split: str = "train"
  seed: int = 3407

  output_dir: str = "outputs"
  lora_save_path: str = "lora_adapters"
  per_device_bs: int = 2
  grad_acc_steps: int = 4
  epochs: int = 1
  lr: float = 2e-5
  warmup_ratio: float = 0.1
  logging_steps: int = 10
  packing: bool = True

  lora_r: int = 32
  lora_alpha: int = 32
  lora_dropout: float = 0.0
  target_modules: List[str] = None
  use_gc: bool = False

  save_merged: bool = False
  merged_save_path: str = "merged_fp16_model"

In [ ]:
class UnslothFineTuner:
  def __init__(self, cfg: FineTuneConfig):
    self.cfg = cfg
    if self.cfg.target_modules is None:
      self.cfg.target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    self.model = None
    self.tokenizer = None

  def load_model(self):
    print(f"Loading model: {self.cfg.model_name}")
    self.model, self.tokenizer = FastLanguageModel.from_pretrained(
        model_name=self.cfg.model_name,
        max_seq_length=self.cfg.max_seq_length,
        load_in_4bit=self.cfg.load_in_4bit,
        dtype=self.cfg.dtype,
    )

    self.model = FastLanguageModel.get_peft_model(
        self.model,
        r=self.cfg.lora_r,
        lora_alpha=self.cfg.lora_alpha,
        lora_dropout=self.cfg.lora_dropout,
        bias="none",
        use_gradient_checkpointing=self.cfg.use_gc,
        random_state=self.cfg.seed,
    )

    self.model.print_trainable_parameters()
    print("Is PEFT model?", isinstance(self.model, PeftModel))
    print("Device:", next(self.model.parameters()).device, " | Dtype:", next(self.model.parameters()).dtype)

  def load_dataset(self):
    print(f"Loading dataset: {self.cfg.dataset_name} (split={self.cfg.split})")

    try:
      ds = load_dataset(self.cfg.dataset_name, split=self.cfg.split)
    except Exception:
      ds = load_dataset(self.cfg.dataset_name, self.cfg.split)

    ds = ds.shuffle(seed=self.cfg.seed)
    print(ds)
    print("Columns:", ds.column_names)
    return ds

  def _eos(self):
    return self.tokenizer.eos_token or ""

  def _alpaca_prompt(self):
    return """Below is an instruction that describes a task, paired with an input that provides further context.
    Write a response that appropriately completes the request.

    ### Instruction:
    {instruction}

    ### Input:
    {input}

    ### Response:
    {output}"""

  def _format_alpaca(self, batch):
    EOS = self._eos()
    prompt = self._alpaca_prompt()
    texts = []
    inp_list = batch.get("input", [""] * len(batch["instruction"]))
    for ins, inp, out in zip(batch["instruction"], inp_list, batch["output"]):
      texts.append(prompt.format(
          instruction=ins,
          input=inp or "",
          output=out or ""
      ) + EOS)
    return {"text": texts}

  def _format_dolly(self, batch):
    EOS = self._eos()
    prompt = self._alpaca_prompt()
    ctx_list = batch.get("context", [""] * len(batch["instruction"]))
    texts = []

    for ins, ctx, resp in zip(batch["instruction"], ctx_list, batch["response"]):
      texts.append(prompt.format(
          instruction=ins or "",
          input=ctx or "",
          output=resp or ""
      ) + EOS)
    return {"text": texts}

  def _format_sharegpt(self, batch):
    EOS = self._eos()
    texts = []

    conv_key = None
    if "conversations" in batch:
      conv_key = "conversations"
    elif "messages" in batch:
      conv_key = "messages"
    else:
      raise ValueError("ShareGPT format needs 'conversations' or 'messages' column.")

    for conv in batch[conv_key]:
      if conv is None:
        texts.append("" + EOS)
        continue

      turns = []
      for t in conv:
        if isinstance(t, dict) and "from" in t and "value" in t:
          role, content = t["from"], t["value"]
        elif isinstance(t, dict) and "role" in t and "content" in t:
          role, content = t["role"], t["content"]
        else:
          continue
        turns.append((role, content))

      chat = ""
      for role, content in turns:
        if role in ["human", "user"]:
          chat += f"### User:\n{content}\n\n"
        else:
          chat += f"### Assistant:\n{content}\n\n"

      texts.append(chat.strip() + EOS)

    return {"text": texts}

  def _format_text(self, batch):
    EOS = self._eos()
    if "text" not in batch:
        raise ValueError("Expected a 'text' column.")
    return {"text": [(t or "") + EOS for t in batch["text"]]}

  def _format_pharma_custom(self, batch):
    EOS = self._eos()
    cols = set(batch.keys())

    if {"instruction", "output"}.issubset(cols):
        return self._format_alpaca(batch)

    if {"question", "answer"}.issubset(cols):
      texts = []
      for q, a in zip(batch["question"], batch["answer"]):
        texts.append(f"### Question:\n{q or ''}\n\n### Answer:\n{a or ''}" + EOS)
      return {"text": texts}

    if {"prompt", "completion"}.issubset(cols):
      texts = []
      for p, c in zip(batch["prompt"], batch["completion"]):
        texts.append(f"{p or ''}\n{c or ''}" + EOS)
      return {"text": texts}

    if {"input", "output"}.issubset(cols):
      prompt = self._alpaca_prompt()
      texts = []
      for i, o in zip(batch["input"], batch["output"]):
        texts.append(prompt.format(
                    instruction="Answer the following:",
                    input=i or "",
                    output=o or ""
                  ) + EOS)
      return {"text": texts}

    raise ValueError(f"Pharma formatter can't infer columns: {sorted(list(cols))}")

  def format_dataset(self, ds):
      print("Inferring dataset format...")

      name = self.cfg.dataset_name
      cols = set(ds.column_names)

      if name == "tatsu-lab/alpaca":
        print("Format: ALPACA")
        return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

      if name == "databricks/databricks-dolly-15k":
        print("Format: DOLLY")
        return ds.map(self._format_dolly, batched=True, remove_columns=ds.column_names)

      if name == "anon8231489123/ShareGPT_Vicuna_unfiltered":
        print("Format: SHAREGPT")
        return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

      if name == "OpenAssistant/oasst1":
        print("Format: OASST (auto)")
        if "text" in cols:
          return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)
        else:
          return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

      if name in ["sunny199/pharma-instruction-data", "pharma_instruction_data"]:
        print(" Format: PHARMA (custom)")
        return ds.map(self._format_pharma_custom, batched=True, remove_columns=ds.column_names)

      if "text" in cols:
        print("Format: TEXT (fallback)")
        return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)

      if "conversations" in cols or "messages" in cols:
        print("Format: CHAT (fallback)")
        return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

      if {"instruction", "output"}.issubset(cols):
          print("Format: ALPACA-LIKE (fallback)")
          return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

      raise ValueError(f"Could not infer dataset format. Columns: {sorted(list(cols))}")

  def train(self, ds):
    trainer = SFTTrainer(
        model=self.model,
        tokenizer=self.tokenizer,
        train_dataset=ds,
        dataset_text_field="text",
        packing=self.cfg.packing,
        args=SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_bs,
            gradient_accumulation_steps=self.cfg.grad_acc_steps,
            num_train_epochs=self.cfg.epochs,
            learning_rate=self.cfg.lr,
            warmup_ratio=self.cfg.warmup_ratio,
            optim="adamw_torch",
            logging_steps=self.cfg.logging_steps,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none"
        ),
    )
    trainer.train()

  def save(self):
    print("Saving LoRA adapters to:", self.cfg.lora_save_path)
    self.model.save_pretrained(self.cfg.lora_save_path)
    self.tokenizer.save_pretrained(self.cfg.lora_save_path)

    if self.cfg.save_merged:
      print("Merging LoRA + saving full model to:", self.cfg.merged_save_path)
      merged = self.model.merge_and_unload()
      merged.save_pretrained(self.cfg.merged_save_path, safe_serialization=True)
      self.tokenizer.save_pretrained(self.cfg.merged_save_path)

  def run(self):
    self.load_model()
    ds = self.load_dataset()
    ds = self.format_dataset(ds)
    print("Sample formatted text:\n", ds["text"][0][:800])
    self.train(ds)
    self.save()

cfg = FineTuneConfig(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    dataset_name="sunny199/pharma-instruction-data",
    split="train",
    lora_save_path="phi3_pharma_lora",
    output_dir="phi3_outputs",
    epochs=1,
    per_device_bs=2,
    grad_acc_steps=4,
    lr=2e-5,
    max_seq_length=4096,
    save_merged=False,
)

trainer = UnslothFineTuner(cfg)
trainer.run()